# Project 11: Workforce Attrition & Compensation Analysis
**Student Name:** Sarthak Dubey  
**Section:** AI ML 5  
**Admission Number:** 2026B15310002

---

**Official Dataset Sources:**
- Kaggle IBM HR Analytics Employee Attrition & Performance: `https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset`
- IBM AIF360 GitHub Raw CSV Mirror: `https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv`

### 1. Problem Statement & Success Criteria
Understand workforce movement, compensation dispersion, and retention indicators across job roles, tenure bands, and overtime patterns using fairness-aware analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style='whitegrid')
os.makedirs('figures', exist_ok=True)

if os.path.exists('data/IBM_HR_Employee_Attrition_Raw.csv'):
    df_raw = pd.read_csv('data/IBM_HR_Employee_Attrition_Raw.csv')
elif os.path.exists('IBM_HR_Employee_Attrition_Raw.csv'):
    df_raw = pd.read_csv('IBM_HR_Employee_Attrition_Raw.csv')
else:
    df_raw = pd.read_csv('https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv')

print(f'Dataset Loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
df_raw.head()

### 2. Data Quality Audit, Grain Validation & Feature Engineering

In [ ]:
assert df_raw['EmployeeNumber'].is_unique, 'Duplicate EmployeeNumber detected!'
print('Primary Key Unique:', df_raw['EmployeeNumber'].is_unique)
print('Total Missing Values:', df_raw.isna().sum().sum())
print('Duplicate Rows:', df_raw.duplicated().sum())

zero_var_cols = df_raw.nunique()[df_raw.nunique() <= 1].index.tolist()
print('Zero-variance columns removed:', zero_var_cols)
df = df_raw.drop(columns=zero_var_cols).copy()

df['AttritionFlag'] = np.where(df['Attrition'] == 'Yes', 1, 0)
df['OverTimeFlag'] = np.where(df['OverTime'] == 'Yes', 1, 0)

df['TenureBand'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1, 2, 5, 10, 45],
    labels=['0-2 yrs (Early)', '3-5 yrs (Mid)', '6-10 yrs (Established)', '10+ yrs (Veteran)']
)

role_stats = df.groupby(['JobRole', 'JobLevel'], as_index=False).agg(
    RoleHeadcount=('EmployeeNumber', 'count'),
    RoleMedianIncome=('MonthlyIncome', 'median'),
    RoleMeanIncome=('MonthlyIncome', 'mean'),
    RoleStdIncome=('MonthlyIncome', 'std')
)
df = df.merge(role_stats, on=['JobRole', 'JobLevel'], how='left', validate='m:1')
assert df['RoleMedianIncome'].isna().sum() == 0

df['CompaRatio'] = np.round(np.divide(df['MonthlyIncome'], df['RoleMedianIncome']), 3)
df['LogMonthlyIncome'] = np.round(np.log1p(df['MonthlyIncome']), 3)
df['PayQuartile'] = df.groupby('JobLevel')['MonthlyIncome'].transform(
    lambda s: pd.qcut(s.rank(method='first'), q=4, labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'])
)
df['StagnationIndex'] = np.round(df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + 1.0), 3)

df_clean = df.drop(columns=['EmployeeNumber'])
print('De-identified analytical table shape:', df_clean.shape)

### 3. NumPy Statistical Dispersion, Outliers & Bootstrap Uncertainty

In [ ]:
inc_all = df_clean['MonthlyIncome'].to_numpy()
q25, q50, q75 = np.percentile(inc_all, [25, 50, 75])
iqr = q75 - q25
upper_fence = q75 + 1.5 * iqr
outlier_mask = inc_all > upper_fence
mad_inc = np.median(np.abs(inc_all - q50))

med_attr = np.median(df_clean.loc[df_clean['AttritionFlag'] == 1, 'MonthlyIncome'].to_numpy())
med_ret = np.median(df_clean.loc[df_clean['AttritionFlag'] == 0, 'MonthlyIncome'].to_numpy())

print(f'Overall Attrition Rate: {df_clean["AttritionFlag"].mean()*100:.1f}% ({df_clean["AttritionFlag"].sum()}/{len(df_clean)})')
print(f'Median Monthly Income: ${q50:,.0f} | IQR: ${iqr:,.0f} | MAD: ${mad_inc:,.0f}')
print(f'High-Income Outliers (> ${upper_fence:,.0f}): {np.sum(outlier_mask)} ({np.mean(outlier_mask)*100:.1f}%)')
print(f'Median Income — Attrited: ${med_attr:,.0f} vs Retained: ${med_ret:,.0f}')

ot_yes = df_clean.loc[df_clean['OverTimeFlag'] == 1, 'AttritionFlag'].to_numpy()
ot_no = df_clean.loc[df_clean['OverTimeFlag'] == 0, 'AttritionFlag'].to_numpy()
rng = np.random.default_rng(42)
boot_diffs = rng.choice(ot_yes, (5000, len(ot_yes)), replace=True).mean(axis=1) - rng.choice(ot_no, (5000, len(ot_no)), replace=True).mean(axis=1)
ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
print(f'OverTime Attrition: {ot_yes.mean()*100:.1f}% vs {ot_no.mean()*100:.1f}% | Lift: +{boot_diffs.mean()*100:.1f} pp (95% CI: [{ci_low*100:.1f}, {ci_high*100:.1f}] pp)')

### 4. pandas Multidimensional Scorecards & Visualisations

In [ ]:
role_sc = df_clean.groupby('JobRole', as_index=False).agg(
    Headcount=('AttritionFlag', 'count'),
    AttritionRate=('AttritionFlag', 'mean'),
    MedianIncome=('MonthlyIncome', 'median'),
    MedianCompaRatio=('CompaRatio', 'median'),
    OverTimeShare=('OverTimeFlag', 'mean')
).sort_values('AttritionRate', ascending=False)
role_sc['AttritionRatePct'] = (role_sc['AttritionRate'] * 100).round(1)
display(role_sc[['JobRole', 'Headcount', 'AttritionRatePct', 'MedianIncome', 'MedianCompaRatio']])

dept_ot_pivot = pd.pivot_table(df_clean, values='AttritionFlag', index='Department', columns='OverTime', aggfunc='mean') * 100
display(dept_ot_pivot.round(1))

overall_attr = df_clean['AttritionFlag'].mean() * 100

plt.figure(figsize=(10, 5))
sns.barplot(data=role_sc, x='AttritionRatePct', y='JobRole', palette='viridis')
plt.axvline(overall_attr, color='crimson', linestyle='--', label=f'Company Avg ({overall_attr:.1f}%)')
plt.title('Figure 1: Voluntary Attrition Rate by Job Role')
plt.xlabel('Attrition Rate (%)')
plt.ylabel('Job Role')
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig1_attrition_by_role.png', dpi=300)
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean, x='JobLevel', y='MonthlyIncome', hue='Attrition', palette='Set2')
plt.title('Figure 2: Monthly Income Distribution by Job Level and Attrition Status')
plt.xlabel('Job Level (1 = Entry, 5 = Executive)')
plt.ylabel('Monthly Income (USD)')
plt.tight_layout()
plt.savefig('figures/fig2_income_by_level_attrition.png', dpi=300)
plt.show()

hm = pd.pivot_table(df_clean, values='AttritionFlag', index='TenureBand', columns='OverTime', aggfunc='mean', observed=False) * 100
plt.figure(figsize=(7, 4.5))
sns.heatmap(hm, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Attrition Rate (%)'})
plt.title('Figure 3: Attrition Rate (%) by Tenure Band and Overtime Status')
plt.xlabel('Overtime Status')
plt.ylabel('Company Tenure Band')
plt.tight_layout()
plt.savefig('figures/fig3_tenure_overtime_heatmap.png', dpi=300)
plt.show()

q_sum = df_clean.groupby('PayQuartile', observed=False)['AttritionFlag'].mean().reset_index()
q_sum['AttritionPct'] = q_sum['AttritionFlag'] * 100
plt.figure(figsize=(8, 4.5))
sns.barplot(data=q_sum, x='PayQuartile', y='AttritionPct', palette='Blues_d')
plt.title('Figure 4: Attrition Rate Across Within-Level Pay Quartiles')
plt.xlabel('Within-Job-Level Compensation Quartile')
plt.ylabel('Attrition Rate (%)')
plt.tight_layout()
plt.savefig('figures/fig4_pay_quartile_attrition.png', dpi=300)
plt.show()

plt.figure(figsize=(9, 5))
sns.violinplot(data=df_clean, x='Department', y='MonthlyIncome', hue='Attrition', split=True, palette='muted')
plt.axhline(upper_fence, color='red', linestyle=':', label=f'IQR Upper Outlier Fence (${upper_fence:,.0f})')
plt.title('Figure 5: Compensation Dispersion and High-Pay Outliers by Department')
plt.xlabel('Department')
plt.ylabel('Monthly Income (USD)')
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig5_dept_income_dispersion.png', dpi=300)
plt.show()